#  Obj Detection and Tracking

In [ ]:
# Install dependencies (run once, if needed)
#import sys
#!{sys.executable} -m pip install --quiet ultralytics norfair opencv-python matplotlib ipywidgets nbformat

In [ ]:
# Import libraries
import cv2
import numpy as np
from ultralytics import YOLO
from norfair import Detection, Tracker
import matplotlib.pyplot as plt
from IPython.display import display
import ipywidgets as widgets
from ipywidgets import interact
import os 

In [ ]:
# Visualization helper (user-provided, slightly adapted)
def showVideo(I):
    n = len(I)
    if n == 0:
        print('No frames to show.')
        return
    def view_image(idx):
        plt.figure(figsize=(10,6))
        img = I[idx-1]
        if img is None:
            plt.text(0.5, 0.5, 'Frame not available', ha='center')
            return
        if len(img.shape) == 3 and img.shape[2] == 3:
            img_rgb = img[..., ::-1]
            plt.imshow(img_rgb)
        else:
            plt.imshow(img, interpolation='nearest', cmap='gray')
        plt.axis('off')
    interact(view_image, idx=widgets.IntSlider(min=1, max=n, step=1, value=1))

In [ ]:
# Load video frames from specified directory
vid_dir = '../data/video/'
 
cap = cv2.VideoCapture(vid_dir + 'IPPR/in000%3d.jpg')
#cap = cv2.VideoCapture(vid_dir + 'los_angeles.mp4' )
frames = []
while True:
    ret, frame = cap.read()
    if not ret:
        break
    frames.append(frame)
cap.release()
 

In [ ]:
# Robust distance function that accepts Norfair Detection/TrackedObject or numpy arrays
import numpy as np
from norfair import Tracker

def robust_distance(a, b):
    """
    Robust distance function for Norfair that accepts either:
      - numpy arrays (points), or
      - Norfair objects (Detection, TrackedObject).
    Returns a scalar distance.
    """
    def to_array(x):
        # Detection: has .points ; TrackedObject: has .estimate
        if hasattr(x, "points"):
            arr = np.asarray(x.points)
        elif hasattr(x, "estimate"):
            arr = np.asarray(x.estimate)
        else:
            arr = np.asarray(x)
        return arr

    pa = to_array(a)
    pb = to_array(b)

    # normalize shapes: if multiple points, reduce to a single representative point (mean)
    if pa.ndim > 1:
        pa = pa.reshape(-1) if pa.shape[0] == 1 else pa.mean(axis=0)
    if pb.ndim > 1:
        pb = pb.reshape(-1) if pb.shape[0] == 1 else pb.mean(axis=0)

    pa = np.asarray(pa, dtype=float).reshape(-1)
    pb = np.asarray(pb, dtype=float).reshape(-1)

    # if shapes differ, compute on overlapping dims
    if pa.shape != pb.shape:
        m = min(pa.size, pb.size)
        return np.linalg.norm(pa[:m] - pb[:m])
    return np.linalg.norm(pa - pb)

# Create tracker with the robust distance function
tracker = Tracker(distance_function=robust_distance, distance_threshold=100)
print('Tracker (robust) created with robust_distance function')


In [ ]:
# Choose YOLO model (will download if not present)
model = YOLO('yolov8n.pt')  # change to yolov8s/m if you have a GPU
#print('Model loaded:', model)

In [ ]:
# --- YOLO -> Norfair conversion helper (DEFINIZIONE NECESSARIA) ---
import numpy as np
from norfair import Detection

def yolo_to_norfair_detections(results, class_filter=None):
    """
    Convert Ultralytics YOLOv8 results (single frame) to Norfair Detection list.
    Uses bbox center as a single tracking point.
    """
    dets = []
    for box in results.boxes:
        # xyxy has shape (1,4)
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        conf = float(box.conf[0])
        cls = int(box.cls[0])

        if class_filter is not None and cls not in class_filter:
            continue

        cx = (x1 + x2) / 2.0
        cy = (y1 + y2) / 2.0

        # Norfair expects (n_points, dim)
        points = np.array([[cx, cy]], dtype=float)
        scores = np.array([conf], dtype=float)

        dets.append(Detection(points=points, scores=scores))

    return dets


In [ ]:
# Processing loop: detection + tracking with robust shapes and debug
processed_frames = []
class_filter = None  # e.g., [0] to track only persons

print('Running detection + tracking over frames...')
for i, frame in enumerate(frames):
    img = frame.copy()
    # Optionally resize for speed: img = cv2.resize(img, (640, 360))
    results = model(img)[0]  # single-frame inference (Ultralytics)
    detections = yolo_to_norfair_detections(results, class_filter=class_filter)
    # Optional debug: print counts for first frames
    if i < 3:
        print(f'Frame {i}: YOLO boxes = {len(results.boxes)}, Norfair dets = {len(detections)}')
    try:
        tracked = tracker.update(detections=detections)
    except Exception as e:
        print('Error in tracker.update on frame', i)
        print('Exception:', repr(e))
        print('Dumping detection shapes:')
        for idx, d in enumerate(detections):
            print(idx, 'points.shape=', getattr(d.points, 'shape', None), 'scores.shape=', getattr(d.scores, 'shape', None))
            print('points=', d.points, 'scores=', d.scores)
        raise

    # Visualization: draw boxes and tracked ids
    vis = img.copy()
    for box in results.boxes:
        xy = box.xyxy[0].cpu().numpy().astype(int)
        x1, y1, x2, y2 = int(xy[0]), int(xy[1]), int(xy[2]), int(xy[3])
        conf = float(box.conf[0]) if hasattr(box.conf, '__len__') else float(box.conf)
        cls = int(box.cls[0]) if hasattr(box.cls, '__len__') else int(box.cls)
        label = f"{model.names[cls]} {conf:.2f}"
        cv2.rectangle(vis, (x1, y1), (x2, y2), (255, 0, 0), 2)
        cv2.putText(vis, label, (x1, max(15, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)

    for to in tracked:
        est = np.asarray(to.estimate).reshape(-1)
        x, y = int(est[0]), int(est[1])
        cv2.circle(vis, (x, y), 6, (0, 255, 0), -1)
        cv2.putText(vis, f'ID {to.id}', (x + 8, y - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    processed_frames.append(vis)
    if (i + 1) % 100 == 0:
        print(f'Processed {i+1}/{len(frames)} frames')

print('Processing complete. Call showVideo(processed_frames) to inspect frames.')

In [ ]:
showVideo(processed_frames)